# AE2 - Reconocimiento del dataset Olist

**Proyecto integrador - Etapa 1: ideación, diseño y documentación de la necesidad**

Este notebook obtiene el dataset desde Hugging Face, lo carga en un DataFrame de pandas, hace una
descripción básica para reconocer su contenido y genera automáticamente un resumen en
`AE2/resultados/resumen_dataframe.txt`. No es un análisis detallado: su propósito es conocer los datos
para formular la necesidad.

**Necesidad (escenario hipotético):** el equipo comercial de un marketplace necesita conocer cómo se
distribuyen las ventas entre categorías de producto y regiones, y cómo se comportan los tiempos de
entrega, para orientar decisiones comerciales y logísticas.

**Pregunta de análisis:** ¿cómo se distribuyen las ventas (unidades e ingresos) por categoría de producto
y por estado del cliente en las órdenes de Olist entre 2016 y 2018, y cómo varía el tiempo de entrega
entre esos grupos?

| Ficha | Valor |
|---|---|
| Nombre | Brazilian E-Commerce Public Dataset by Olist (copia en Hugging Face) |
| Identificador | `bulutttt/olist-raw-data` |
| Enlace | https://huggingface.co/datasets/bulutttt/olist-raw-data |
| Revisión utilizada | `5acf1f37d7197d4979e714721a1ec13a7c3da728` |
| Split | `train` (único disponible) |
| Fecha de consulta | 2026-10-25 |
| Tipo de datos | Reales y anonimizados (no sintéticos) |
| Muestra | No: se usan todos los registros de las tablas cargadas |

## 1. Preparar el entorno e importar las clases

Se agrega `src` a la ruta de Python para importar las clases reutilizables del proyecto:
`HFDatasetLoader` (obtención de datos) y `DatasetSummarizer` (resumen en `.txt`).

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

import pandas as pd
from datasets.utils.logging import disable_progress_bar, set_verbosity_error
from huggingface_hub.utils import logging as hf_logging

# Silenciar avisos y barras de progreso de la descarga
set_verbosity_error()
hf_logging.set_verbosity_error()
disable_progress_bar()

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases_po").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases_po import DatasetSummarizer, HFDatasetLoader

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

## 2. Obtener los datos desde Hugging Face

El repositorio publica cada tabla como un `.csv`. Se cargan las tablas necesarias para la necesidad,
fijando la revisión del repositorio para que el resultado sea reproducible. No se cargan
`olist_geolocation_dataset.csv` (coordenadas por código postal) ni `olist_order_reviews_dataset.csv`
(reseñas), porque no se requieren para la pregunta planteada.

In [2]:
DATASET_ID = "bulutttt/olist-raw-data"
REVISION = "5acf1f37d7197d4979e714721a1ec13a7c3da728"
SPLIT = "train"
ARCHIVOS = {
    "items": "olist_order_items_dataset.csv",
    "ordenes": "olist_orders_dataset.csv",
    "productos": "olist_products_dataset.csv",
    "categorias": "product_category_name_translation.csv",
    "clientes": "olist_customers_dataset.csv",
    "vendedores": "olist_sellers_dataset.csv",
}

tablas = HFDatasetLoader.load_tables(DATASET_ID, ARCHIVOS, split=SPLIT, revision=REVISION)

pd.DataFrame(
    {
        "archivo": ARCHIVOS.values(),
        "filas": [len(df) for df in tablas.values()],
        "columnas": [df.shape[1] for df in tablas.values()],
    },
    index=ARCHIVOS.keys(),
)

,archivo,filas,columnas
items,olist_order_items_dataset.csv,112650,7
ordenes,olist_orders_dataset.csv,99441,8
productos,olist_products_dataset.csv,32951,9
categorias,product_category_name_translation.csv,71,2
clientes,olist_customers_dataset.csv,99441,5
vendedores,olist_sellers_dataset.csv,3095,4


## 3. Construir el DataFrame de análisis

La **unidad de análisis** es el **ítem de orden**: cada fila de `olist_order_items_dataset.csv` es un
producto vendido dentro de una orden, con su precio y su flete. A cada ítem se le agregan, mediante
uniones por llave, los datos de su orden (estado y fechas), del producto (categoría), del cliente
(ubicación) y del vendedor (ubicación). Se usa `left join` desde los ítems para no perder ni duplicar
registros: el DataFrame final debe tener exactamente las mismas filas que la tabla de ítems.

In [3]:
df = (
    tablas["items"]
    .merge(tablas["ordenes"], on="order_id", how="left")
    .merge(tablas["productos"], on="product_id", how="left")
    .merge(tablas["categorias"], on="product_category_name", how="left")
    .merge(tablas["clientes"], on="customer_id", how="left")
    .merge(tablas["vendedores"], on="seller_id", how="left")
)

assert len(df) == len(tablas["items"]), "Las uniones duplicaron o perdieron filas"
print(f"Ítems: {len(tablas['items'])} | DataFrame de análisis: {len(df)} filas")

Ítems: 112650 | DataFrame de análisis: 112650 filas


## 4. Descripción básica del DataFrame

### 4.1 Algunos registros

In [4]:
df.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,seller_zip_code_prefix,seller_city,seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29 00:00:00,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,cool_stuff,871766c5855e863f6eccc05f988b23cb,28013,campos dos goytacazes,RJ,27277,volta redonda,SP
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15 00:00:00,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,pet_shop,eb28e67c4c0b83846050ddfb8a35d051,15775,santa fe do sul,SP,3471,sao paulo,SP
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05 00:00:00,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,furniture_decor,3818d81c6709e39d06b2738a8d3a2474,35661,para de minas,MG,37564,borda da mata,MG
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20 00:00:00,perfumaria,42.0,480.0,1.0,200.0,16.0,10.0,15.0,perfumery,af861d436cfc08b2c2ddefd0ba074622,12952,atibaia,SP,14403,franca,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17 00:00:00,ferramentas_jardim,59.0,409.0,1.0,3750.0,35.0,40.0,30.0,garden_tools,64b576fb70d441e8f1b2d7d446e483c5,13226,varzea paulista,SP,87900,loanda,PR


In [5]:
df.sample(5, random_state=42)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,seller_zip_code_prefix,seller_city,seller_state
107777,f4ee4273538924bda6212f5948e80fde,1,69455f41626a745aea9ee9164cb9eafd,7d13fca15225358621be4086e1eb0964,2018-06-19 09:18:15,180.00,20.45,8871d0db2013a98a58576b50ff9a1c1d,delivered,2018-06-12 08:59:54,2018-06-12 09:18:15,2018-06-15 16:36:00,2018-06-25 16:08:40,2018-07-13 00:00:00,relogios_presentes,41.0,408.0,2.0,501.0,18.0,13.0,14.0,watches_gifts,8cebd890b95f34cd04b08f958c1af21c,28943,sao pedro da aldeia,RJ,14050,ribeirao preto,SP
2391,056349f85a73d794119c4286c95a52de,1,af35be35db4ad0dc288b571453337376,d20b021d3efdf267a402c402a48ea64b,2017-03-09 14:35:09,10.99,16.05,03bbe0ce5c28e05f22917607db798818,delivered,2017-03-03 14:21:58,2017-03-03 14:35:09,2017-03-03 16:08:50,2017-03-17 12:52:17,2017-03-28 00:00:00,moveis_decoracao,24.0,457.0,4.0,100.0,80.0,10.0,60.0,furniture_decor,8f3dca4306d5a89e4ae2c65c110603a2,72465,brasilia,DF,14940,ibitinga,SP
77829,b124967afcc82ef17ec41020fe2a9136,1,12e6d0f655986ceff00c74658dec97b1,3be634553519fb6536a03e1358e9fdc7,2018-06-06 17:18:09,49.99,8.88,43dd5cdd60809c5f268d9d3b1e5b986d,delivered,2018-05-29 16:19:32,2018-05-29 17:18:09,2018-05-30 14:09:00,2018-06-01 15:48:34,2018-06-21 00:00:00,moveis_sala,47.0,511.0,1.0,7875.0,60.0,25.0,25.0,furniture_living_room,da6ad3471341a3f6801cb11ea9a88b3c,5311,sao paulo,SP,8275,sao paulo,SP
99819,e257ae8610fb4fb68a1f459c3a4b1f51,1,a50acd33ba7a8da8e9db65094fa990a4,8581055ce74af1daba164fdbd55a40de,2017-05-18 08:02:19,117.30,14.43,b46f11e377a570f09af36515191b1c1e,delivered,2017-05-13 07:49:29,2017-05-13 08:02:19,2017-05-16 08:39:35,2017-05-19 07:29:09,2017-05-24 00:00:00,automotivo,53.0,555.0,1.0,4105.0,67.0,10.0,67.0,auto,6d417685c0ddaca4f3926f98d0601768,9170,santo andre,SP,7112,guarulhos,SP
41297,5e114d8e3840661abc3d9c4820f427b3,1,5cca3efb9521cc1d7099d610d4a12017,3d871de0142ce09b7081e2b9d1733cb1,2018-05-03 18:15:09,58.90,13.77,b7bf7024fbd5f37207415a82f73d70e6,delivered,2018-04-26 18:01:20,2018-04-26 18:15:09,2018-04-30 07:37:00,2018-05-04 18:19:56,2018-05-17 00:00:00,brinquedos,60.0,176.0,3.0,5700.0,39.0,30.0,29.0,toys,16fa21bb47836bc052c05c338032359f,9715,sao bernardo do campo,SP,13232,campo limpo paulista,SP


### 4.2 Dimensiones

In [6]:
filas, columnas = df.shape
print(f"Filas: {filas}")
print(f"Columnas: {columnas}")

Filas: 112650
Columnas: 30


### 4.3 Columnas, tipos de datos y valores nulos

Las fechas llegan como texto (`object`); se convertirán a fecha en la siguiente etapa del proyecto.

In [7]:
descripcion_columnas = pd.DataFrame(
    {
        "tipo": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "% nulos": (df.isna().mean() * 100).round(2),
        "valores_unicos": df.nunique(),
    }
)
descripcion_columnas

,tipo,nulos,% nulos,valores_unicos
order_id,object,0,0.00,98666
order_item_id,int64,0,0.00,21
product_id,object,0,0.00,32951
seller_id,object,0,0.00,3095
shipping_limit_date,object,0,0.00,93318
price,float64,0,0.00,5968
freight_value,float64,0,0.00,6999
customer_id,object,0,0.00,98666
order_status,object,0,0.00,7
order_purchase_timestamp,object,0,0.00,98112


### 4.4 Variables numéricas y categóricas principales

Una mirada inicial a las variables que respaldan la necesidad: precio, flete, estado de la orden,
categoría y estado del cliente.

In [8]:
df[["price", "freight_value", "product_weight_g"]].describe().round(2)

,price,freight_value,product_weight_g
count,112650.00,112650.00,112632.00
mean,120.65,19.99,2093.67
std,183.63,15.81,3751.60
min,0.85,0.00,0.00
25%,39.90,13.08,300.00
50%,74.99,16.26,700.00
75%,134.90,21.15,1800.00
max,6735.00,409.68,40425.00


In [9]:
df["order_status"].value_counts()

order_status
delivered      110197
shipped          1185
canceled          542
invoiced          359
processing        357
unavailable         7
approved            3
Name: count, dtype: int64

In [10]:
df["product_category_name_english"].value_counts().head(10)

product_category_name_english
bed_bath_table           11115
health_beauty             9670
sports_leisure            8641
furniture_decor           8334
computers_accessories     7827
housewares                6964
watches_gifts             5991
telephony                 4545
garden_tools              4347
auto                      4235
Name: count, dtype: int64

In [11]:
df["customer_state"].value_counts().head(10)

customer_state
SP    47449
RJ    14579
MG    13129
RS     6235
PR     5740
SC     4176
BA     3799
DF     2406
GO     2333
ES     2256
Name: count, dtype: int64

In [12]:
print(f"Periodo de compras: {df['order_purchase_timestamp'].min()} -> {df['order_purchase_timestamp'].max()}")
print(f"Órdenes distintas: {df['order_id'].nunique()}")
print(f"Clientes distintos: {df['customer_unique_id'].nunique()}")
print(f"Vendedores distintos: {df['seller_id'].nunique()}")
print(f"Productos distintos: {df['product_id'].nunique()}")

Periodo de compras: 2016-09-04 21:15:19 -> 2018-09-03 09:06:57
Órdenes distintas: 98666
Clientes distintos: 95420
Vendedores distintos: 3095
Productos distintos: 32951


## 5. Qué representa cada fila y qué variables respaldan la necesidad

**Cada fila es un producto vendido dentro de una orden** (ítem de orden). Una orden con tres productos
aparece en tres filas, por eso hay más filas que órdenes distintas.

| Parte de la pregunta | Variables del DataFrame |
|---|---|
| Ingresos por venta | `price` (precio del ítem, en reales brasileños), `freight_value` (flete) |
| Unidades vendidas | Conteo de filas (cada fila es un ítem vendido) |
| Categoría de producto | `product_category_name_english` (categoría traducida), `product_category_name` |
| Región | `customer_state`, `customer_city`, `seller_state` |
| Periodo | `order_purchase_timestamp` (fecha de compra) |
| Tiempo de entrega | `order_purchase_timestamp`, `order_delivered_customer_date`, `order_estimated_delivery_date` |
| Ventas efectivas | `order_status` (permite separar órdenes entregadas de canceladas o no disponibles) |

Observaciones para el diseño:

- `order_delivered_customer_date` tiene nulos: corresponden a órdenes no entregadas
  (canceladas, en tránsito, etc.). El tiempo de entrega solo se podrá calcular para órdenes entregadas.
- `product_category_name_english` tiene nulos: productos sin categoría en el origen o categorías sin
  traducción. Se tratarán como "sin categoría".

## 6. Generar automáticamente el resumen en `.txt`

`DatasetSummarizer` recibe el mismo DataFrame descrito arriba y escribe el resumen en
`AE2/resultados/resumen_dataframe.txt` (ruta relativa al notebook). La carpeta se crea si no existe y
el archivo se regenera en cada ejecución, con codificación UTF-8.

In [13]:
RUTA_RESUMEN = Path("..") / "resultados" / "resumen_dataframe.txt"

metadatos = {
    "Dataset": DATASET_ID,
    "Enlace": f"https://huggingface.co/datasets/{DATASET_ID}",
    "Revisión": REVISION,
    "Split": SPLIT,
    "Archivos": ", ".join(ARCHIVOS.values()),
    "Muestra": "No, se usaron todos los registros de los archivos cargados",
    "Unidad de análisis": "Ítem de orden (un producto vendido dentro de una orden)",
}

resumen = DatasetSummarizer(df, titulo="Resumen del DataFrame - Olist (AE2)", metadatos=metadatos)
ruta = resumen.guardar(RUTA_RESUMEN)
print(f"Resumen guardado en: {ruta.resolve().relative_to(ROOT)}")

Resumen guardado en: AE2\resultados\resumen_dataframe.txt


### 6.1 Verificar el contenido del archivo generado

Se lee el archivo y se comprueba que las dimensiones registradas coinciden con el DataFrame.

In [14]:
contenido = ruta.read_text(encoding="utf-8")

assert f"Filas: {df.shape[0]}" in contenido
assert f"Columnas: {df.shape[1]}" in contenido
assert all(columna in contenido for columna in df.columns)
print("Verificación correcta: el resumen corresponde al DataFrame analizado.\n")

print(contenido)

Verificación correcta: el resumen corresponde al DataFrame analizado.

RESUMEN DEL DATAFRAME - OLIST (AE2)
Generado: 2026-10-25 13:29:58

Origen de los datos
-------------------
- Dataset: bulutttt/olist-raw-data
- Enlace: https://huggingface.co/datasets/bulutttt/olist-raw-data
- Revisión: 5acf1f37d7197d4979e714721a1ec13a7c3da728
- Split: train
- Archivos: olist_order_items_dataset.csv, olist_orders_dataset.csv, olist_products_dataset.csv, product_category_name_translation.csv, olist_customers_dataset.csv, olist_sellers_dataset.csv
- Muestra: No, se usaron todos los registros de los archivos cargados
- Unidad de análisis: Ítem de orden (un producto vendido dentro de una orden)

Dimensiones
-----------
Filas: 112650
Columnas: 30
Filas duplicadas: 0
Memoria aproximada: 144,216.7 KB

Columnas y tipos
----------------
- order_id: object (98666 valores únicos)
- order_item_id: int64 (21 valores únicos)
- product_id: object (32951 valores únicos)
- seller_id: object (3095 valores únicos)
- s